# NSynth + AFTER Representation Evaluation: Informativeness

This notebook evaluates the **informativeness** of AFTER timbre embeddings extracted from NSynth.

The goal is to train a probe on top of frozen embeddings and evaluate whether a factor of variation, initially `instrument_family`, can be predicted from the embedding space.

This notebook assumes that:

1. The reduced NSynth dataset has already been created.
2. Each split contains a valid `examples.json`.
3. AFTER timbre embeddings have already been extracted and saved as `.pt` files.
4. The folder structure is:

```text
/content/drive/MyDrive/datasets/nsynth/
├── train/
│   ├── *.wav
│   ├── examples.json
│   └── AFTER_Timbre/
│       └── *.pt
├── validation/
│   ├── *.wav
│   ├── examples.json
│   └── AFTER_Timbre/
│       └── *.pt
└── test/
    ├── *.wav
    ├── examples.json
    └── AFTER_Timbre/
        └── *.pt
```


## 1. Mount Drive and configure paths

In [10]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys

REPO_ROOT = Path('/content/drive/MyDrive/SYNESIS/synesis')
DATASET_ROOT = Path('/content/drive/MyDrive/datasets/nsynth')

assert REPO_ROOT.exists(), f'No encuentro el repo en {REPO_ROOT}'
assert DATASET_ROOT.exists(), f'No encuentro NSynth en {DATASET_ROOT}'

os.chdir(REPO_ROOT)

# Make sure Python can import config.features and synesis.*
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('Repo actual:', Path.cwd())
print('Dataset:', DATASET_ROOT)
print('config/features.py existe:', (REPO_ROOT / 'config' / 'features.py').exists())
print('synesis package existe:', (REPO_ROOT / 'synesis').exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo actual: /content/drive/MyDrive/SYNESIS/synesis
Dataset: /content/drive/MyDrive/datasets/nsynth
config/features.py existe: True
synesis package existe: True


## 2. Check Python / NumPy environment

In previous runs, NumPy 2.x caused import issues with this environment.  
For this pipeline, `numpy==1.26.4` was the stable version.


In [2]:
import numpy as np
print('NumPy version:', np.__version__)

if not np.__version__.startswith('1.26'):
    print('WARNING: This environment is not using NumPy 1.26.x.')
    print('If imports fail, run:')
    print('!pip install --force-reinstall "numpy==1.26.4"')
    print('Then restart the runtime.')

NumPy version: 2.0.2
If imports fail, run:
!pip install --force-reinstall "numpy==1.26.4"
Then restart the runtime.


In [1]:
!pip install -q --force-reinstall "numpy==1.26.4"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which

In [2]:
import numpy as np
print(np.__version__)

1.26.4


## 3. Configure the evaluation task

In [3]:
# Feature folder containing the extracted embeddings
FEATURE = 'AFTER_Timbre'

# Factor of Variation / label to predict from the embeddings.
# First experiment: single-label classification of instrument family.
FV = 'instrument_family'

# Synesis task name.
# If the CLI accepts task names, this makes the experiment explicit.
TASK = 'instrument_family_classification'

# Batch size for probe training.
# This is separate from the batch size used during embedding extraction.
BATCH_SIZE_TRAIN = 64

# Reproducibility
SEED = 42

print('Feature:', FEATURE)
print('Label / FV:', FV)
print('Task:', TASK)
print('Batch size train:', BATCH_SIZE_TRAIN)

Feature: AFTER_Timbre
Label / FV: instrument_family
Task: instrument_family_classification
Batch size train: 64


## 4. Verify that embeddings exist

This step checks that each split contains `.pt` files inside the `AFTER_Timbre` folder.


In [6]:
for split in ['train', 'validation', 'test']:
    split_dir = DATASET_ROOT / split
    feature_dir = split_dir / FEATURE
    json_path = split_dir / 'examples.json'
    wav_count = len(list(split_dir.glob('*.wav')))
    pt_count = len(list(feature_dir.glob('*.pt'))) if feature_dir.exists() else 0

    print(f'\nSplit: {split}')
    print('Split dir exists:', split_dir.exists())
    print('examples.json exists:', json_path.exists())
    print('Feature dir exists:', feature_dir.exists())
    print('Number of wav files:', wav_count)
    print('Number of pt embeddings:', pt_count)

    assert split_dir.exists(), f'Missing split directory: {split_dir}'
    assert json_path.exists(), f'Missing examples.json: {json_path}'
    assert feature_dir.exists(), f'Missing feature directory: {feature_dir}'
    assert pt_count > 0, f'No .pt embeddings found in {feature_dir}'


Split: train
Split dir exists: True
examples.json exists: True
Feature dir exists: True
Number of wav files: 11741
Number of pt embeddings: 11741

Split: validation
Split dir exists: True
examples.json exists: True
Feature dir exists: True
Number of wav files: 2516
Number of pt embeddings: 2516

Split: test
Split dir exists: True
examples.json exists: True
Feature dir exists: True
Number of wav files: 2517
Number of pt embeddings: 2517


## 5. Verify that the custom Nsynth dataset loads embeddings correctly

This checks that the dataset can load `.pt` embeddings and read the selected label.


In [13]:
from config.features import configs as feature_configs
from synesis.datasets.nsynth import NSynth

LABEL = "instrument_family"  # o "qualities"

feature_config = {
    **feature_configs[FEATURE],
    "item_len_sec": 4,
    "sample_rate": 44100,
}

ds_train = NSynth(
    feature=FEATURE,
    root=DATASET_ROOT,
    split="train",
    item_format="feature",
    label=LABEL,
    feature_config=feature_config,
)

print("Number of train examples:", len(ds_train))

if LABEL == "qualities":
    print("Multi-label task. Label shape:", ds_train.labels.shape)
else:
    print("Classes:", list(ds_train.label_encoder.classes_))

x, y = ds_train[0]
print("Embedding type:", type(x))
print("Embedding shape:", x.shape if hasattr(x, "shape") else "No direct shape")
print("Label:", y)

FileNotFoundError: Could not find NSynth metadata file: /content/drive/MyDrive/datasets/nsynth/nsynth-train/examples.json

## 6. Check Synesis informativeness CLI

Before launching the experiment, inspect the available command-line options.

Look for options related to:
- feature (`-f`)
- dataset (`-d`)
- label (`-l`)
- task (`-t`)
- batch size
- epochs
- W&B / wandb logging


In [5]:
!pip install -q torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 23.3 MB/s eta 0:00:00


In [6]:
!pip install -q mir_eval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 3.3 MB/s eta 0:00:00


In [7]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA version:", torch.version.cuda)

CUDA available: True
Torch CUDA version: 12.8


In [1]:
!python -m synesis.informativeness.downstream \
    -f AFTER_Timbre \
    -d Nsynth \
    -t classification_SLP \
    -l instrument_family

/usr/bin/python3: Error while finding module specification for 'synesis.informativeness.downstream' (ModuleNotFoundError: No module named 'synesis')


## 7. Optional: connect to Weights & Biases

Run this if you want the probe training to be logged to W&B.

Whether this is enough depends on whether `synesis.informativeness.downstream` already supports W&B flags.  
If the CLI does not support W&B directly, the Synesis training script will need a small modification.


In [ ]:
!pip install -q wandb

import wandb
wandb.login()

## 8. Run informativeness / downstream probing

This command trains a probe to predict `instrument_family` from the frozen `AFTER_Timbre` embeddings.

If `--help` shows different argument names, adjust the command accordingly.


In [ ]:
!python -m synesis.informativeness.downstream \
    -f $FEATURE \
    -d Nsynth \
    -l $FV \
    -t $TASK

## 9. Alternative command template with possible W&B flags

Use this only if the `--help` output shows that the downstream script supports W&B arguments.

The exact flag names may be different, so check the previous help cell first.


In [ ]:
# Example only. Uncomment and adapt if the Synesis CLI supports these flags.
# !python -m synesis.informativeness.downstream \
#     -f $FEATURE \
#     -d Nsynth \
#     -l $FV \
#     -t $TASK \
#     --wandb \
#     --project tfg-after-nsynth

## 10. Notes for future `qualities` experiment

`instrument_family` is a **single-label** classification task.

`qualities` is different: it is a **multi-label** task, because one sound can have several qualities at the same time.

For `qualities`, the probe training code needs:
- output dimension equal to the number of quality labels,
- `BCEWithLogitsLoss`,
- multi-label metrics such as micro-F1 / macro-F1.

The embeddings do **not** need to be re-extracted. Only the probing task changes.


In [ ]:
# Future configuration for qualities:
# FV = 'qualities'
# TASK = 'qualities_classification'
#
# Then the downstream script must support multi-label classification.
# If not, we will need to adapt the training code.